# 第 7 章 · 多 agent（Parallel Processing）

两个 agent **同时**跑，各写各的 state 键，最后合起来看。

和 Sequential 的唯一区别是并发：它们之间不能有数据依赖。

kernel 选 `Python (agentic-adk)`。

In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）

# 通用脚手架都在 common 里：run_once / run_session / run_async / 重试 / 静音
from common import (adk_model, quiet_adk_sync_warnings, retry_call,
                    run_once, run_session, run_async, require_adk)
quiet_adk_sync_warnings()

require_adk()      # 【适配】内核选错时给出可操作提示（内部就是 import google.adk）

from google.adk.agents import LlmAgent, ParallelAgent

weather_fetcher = LlmAgent(
    name="weather_fetcher",
    model=adk_model(),                      # 【适配】gemini-2.0-flash-exp 已下线
    instruction="给出所给地点的天气，只回一句话。",
    output_key="weather_data",
)

news_fetcher = LlmAgent(
    name="news_fetcher",
    model=adk_model(),                      # 【适配】同上
    instruction="给出所给主题的一条新闻标题和一句话摘要。",
    output_key="news_data",
)

# ★ 本章新增 ParallelAgent：子 agent 同时启动，全部结束后才算自己结束
data_gatherer = ParallelAgent(name="data_gatherer", sub_agents=[weather_fetcher, news_fetcher])

finals, state = run_once(data_gatherer, "地点：北京；主题：大型户外运动活动")
for author, text in finals:
    print(f"\n--- [{author}] ---\n{text[:300]}")
print("\n--- 汇合后的 state ---")
print("weather_data:", (state.get("weather_data") or "(空)")[:200])
print("news_data   :", (state.get("news_data") or "(空)")[:200])


--- [weather_fetcher] ---
今天北京天气晴朗，气温适宜，微风拂面，非常适合举办大型户外运动活动。

--- [news_fetcher] ---
**新闻标题：** 
2024北京户外运动节在延庆盛大开幕，掀起京郊“体育+文旅”消费新热潮

**一句话摘要：** 
本届户外运动节集结了越野、骑行、露营等多种时尚户外项目，吸引了数万名运动爱好者齐聚绿水青山间，有力推动了北京文体旅产业的深度融合。

--- 汇合后的 state ---
weather_data: 今天北京天气晴朗，气温适宜，微风拂面，非常适合举办大型户外运动活动。
news_data   : **新闻标题：** 
2024北京户外运动节在延庆盛大开幕，掀起京郊“体育+文旅”消费新热潮

**一句话摘要：** 
本届户外运动节集结了越野、骑行、露营等多种时尚户外项目，吸引了数万名运动爱好者齐聚绿水青山间，有力推动了北京文体旅产业的深度融合。
